## AI classification DOB complaint data

In [2]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()              # makes pretty progress bars in pandas with progress_apply()

# this should be familiar!
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENROUTER_API_KEY_LEDE"][:8] + "..."

'sk-or-v1...'

In [12]:
df = pd.read_csv('complaint_detail.csv')
df = df.dropna(subset=['comments'])
text_column_name = "comments"
df
#results = []

#for comment in df['comments']:
#    category = classify_commnet(comment)
#    results.append(category)

#df['category'] = results

,complaint_no,title,complaint_at,bin,owner,disposition,comments,disp_date,disp_code,disp_result,year
0,1725087,Overview for Complaint #:1725087 = RESOLVED,1251 AVENUE OF THE AMERICAS,1022706.0,UNAVAILABLE OWNER,06/23/2026 - A8 - ECB VIOLATION SERVED,DOUBLE-KNOT RESTAURANT HAS EXPANDED ONTO THE P...,2026-06-23,A8,ECB VIOLATION SERVED,2026.0
1,1724719,Overview for Complaint #:1724719 = RESOLVED,301 EAST 79 STREET,1049261.0,CONTINENTAL TOWERS CONDO,06/03/2026 - A8 - ECB VIOLATION SERVED,POPS SIGNAGE DOESN'T CONFORM TO DCP REGULATION...,2026-06-03,A8,ECB VIOLATION SERVED,2026.0
2,1725403,Overview for Complaint #:1725403 = RESOLVED,118 EAST 60 STREET,1041902.0,118 E 60 OWNERS INC,05/18/2026 - A8 - ECB VIOLATION SERVED,POPS SITE REQUIRES SIGNAGE APPROVED BY DCP POS...,2026-05-18,A8,ECB VIOLATION SERVED,2026.0
3,1724808,Overview for Complaint #:1724808 = RESOLVED,422 EAST 72 STREET,1045832.0,OXFORD CONDO C/O B H S,05/18/2026 - A8 - ECB VIOLATION SERVED,POPS SIGNAGE IS SUBSTANTIALLY BLOCKED BY PROPE...,2026-05-18,A8,ECB VIOLATION SERVED,2026.0
4,1725121,Overview for Complaint #:1725121 = RESOLVED,630 1 AVENUE,1022060.0,ANTHONY PAUL GIORGIO,05/14/2026 - A8 - ECB VIOLATION SERVED,"POPS SITE, FAILURE TO COMPLY WITH DCP REQUIRED...",2026-05-14,A8,ECB VIOLATION SERVED,2026.0
...,...,...,...,...,...,...,...,...,...,...,...
455,1082005,Overview for Complaint #:1082005 = RESOLVED,205 EAST 95 STREET,1049245.0,YORKVILLE PLAZA ASSOC,11/20/2000 - A1 - BUILDINGS VIOLATION(S) SERVED,ONLY THREE BICYCLE PARKING SPACES PROVIDED IN ...,2000-11-20,A1,BUILDINGS VIOLATION(S) SERVED,2000.0
456,1081998,Overview for Complaint #:1081998 = RESOLVED,1 LINCOLN PLAZA,1027472.0,BOUSBIB ARI,11/20/2000 - A1 - BUILDINGS VIOLATION(S) SERVED,FAILURE TO COMPLY WITH AS-OF-RIGHT PLAZA BONUS,2000-11-20,A1,BUILDINGS VIOLATION(S) SERVED,2000.0
457,1082008,Overview for Complaint #:1082008 = RESOLVED,235 EAST 95 STREET,1083228.0,M F ASSOCIATES,11/20/2000 - A1 - BUILDINGS VIOLATION(S) SERVED,"ONLY 4 BICYCLE PARKING SPACES PROVIDED, 4 CU.F...",2000-11-20,A1,BUILDINGS VIOLATION(S) SERVED,2000.0
458,1082013,Overview for Complaint #:1082013 = RESOLVED,141 EAST 48 STREET,1036233.0,NORMAN FOUNDATION,11/20/2000 - A1 - BUILDINGS VIOLATION(S) SERVED,FAILURE TO PROVIDE AMENITIES IN VIOLATION OF A...,2000-11-20,A1,BUILDINGS VIOLATION(S) SERVED,2000.0


In [6]:
## setting up our connection to an API client for an AI provider
# for Lede, we're using OpenRouter, which connects to most providers
# but I've provided code for connect to other providers directly, if you want to do that in real life

USE_OPENROUTER=True

from anthropic import Anthropic

if USE_OPENROUTER:
  from openrouter import OpenRouter
else:
  from openai import OpenAI
  from mistralai.client import Mistral

if USE_OPENROUTER:
  openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))
else:
  openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
  mistral_client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))
anthropic_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"),) # intentionally created outside of the if; we'll use anthropic_client to count tokens (even if we're using openrouter)

In [13]:
## prompt
## start with the one YOU tried in the LLM web interface
## don't worry, we'll adjust it later.
## Jeremy's is at the very bottom, if you need to reference it, but please come
##   up with your own.


# Be sure to list the categories (defined below!) with the magic phrase {categories}
prompt_base = """

Please read this comment regarding the POPS violation, and tell me which category best describes
it, choosing from just these options: {categories}. Return only the topic name,
nothing else.


Here is the comment: {comment_text}

"""

from enum import Enum, IntEnum

# force the AI to ONLY guess fromn the options we ask you.
# you can change these names! (and perhaps should!)
# - the AI sees the _value_, not the key (it sees "broader_political_issues_except_us_election", not "other_politics")
# - the key is just for you -- a shorthand
# - some AIs have trouble with complicated names, so it's better to stick to lowercase, no special characters
#     (as of early June 2026 -- maybe this will be fixed)
class POPSOptions(str, Enum):
    signage = 'signage'
    amenities = 'amenities'
    commercial_encroachment = 'commercial_encroachment'
    access_restriction = 'access_restriction'
    maintenance = 'maintenance'

# for convenience, make a column in our dataframes with the prompt for each item
df_prompt_column = df.apply(lambda row: prompt_base.format(
    comment_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in POPSOptions])
), axis="columns")

In [25]:
df_prompt_column.iloc[350]

'\n\nPlease read this comment regarding the POPS violation, and tell me which category best describes\nit, choosing from just these options: "signage", "amenities", "commercial_encroachment", "access_restriction", "maintenance". Return only the topic name,\nnothing else.\n\n\nHere is the comment: URBAN PLAZA PRIVATIZED BY "MILOS CAFE NEW YORK" WITH TABLES                     & CHAIRS .METAL RAILS INSTALLED ON EAST PLANTER LEDGES\n\n'

In [15]:
## cost estimation

from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-4.5-haiku": 1 / 1_000_000,
    "claude-4.6-sonnet": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [20]:
# IMPORTANT: let's pick a model to use
# it's got to be listed in INPUT_TOKEN_COSTS up above

DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE


Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-4.5-haiku
 - claude-4.6-sonnet

type a model name (press enter for default: gpt-5.4-nano):  gpt-5.5


In [21]:
## cost estimation

count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(df_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.25'

In [22]:
## a lot of boilerplate for sending our prompt + tweet to the model and getting an answer
from pydantic import BaseModel


class POPSValidOptions(BaseModel):
  classification: POPSOptions

SYSTEM_PROMPT = "You are a specialist of NYC's POPS"

def anthropic_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    message = anthropic_client.messages.parse(
        max_tokens=1024,
        # system=SYSTEM_PROMPT, # causes trouble with structured outputs, oddly
        messages = [
            {
                "role": "user",
                "content": prompt_including_comment,
            }
        ],
        model=MODEL_TO_USE,
        output_format=POPSValidOptions
    )
    return message.content[0].parsed_output.classification.value if message.content[0].parsed_output else None

def mistral_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    chat_response = mistral_client.chat.parse(
      model=MODEL_TO_USE,
      messages=[
          {
              "role": "system",
              "content": SYSTEM_PROMPT
          },
          {
              "role": "user",
              "content": prompt_including_comment
          },
      ],
      response_format=POPSValidOptions,
      max_tokens=256,
      temperature=0
    )
    return chat_response.choices[0].message.parsed.classification.value if chat_response.choices[0].message.parsed else None

def chatgpt_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_comment,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=POPSValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-4.5-haiku": "anthropic/claude-4.5-haiku",
    "claude-4.6-sonnet": "anthropic/claude-4.6-sonnet"
}
def openrouter_classify(prompt_including_comment):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_comment}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "comment_classification",
                "strict": True,
                "schema": POPSValidOptions.model_json_schema(),
            },
        },
    )
    return  POPSValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_comment):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_comment)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_comment)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_comment)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_comment)
    else:
      raise ValueError("Unknown model")

In [27]:
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
df["ai_guess"] = df_prompt_column.progress_apply(classify)

100%|███████████████████████████████████████| 460/460 [1:07:32<00:00,  8.81s/it]


In [34]:
df = df[['complaint_no', 'title', 'complaint_at', 'bin', 'owner', 'disposition',
       'disp_date', 'disp_code', 'disp_result', 'year', 'comments',
       'ai_guess']]
df.to_csv('AI_clasification_complaint.csv', index=False)

In [41]:
df['ai_guess'].value_counts()

ai_guess
amenities                  153
signage                    101
access_restriction          97
commercial_encroachment     73
maintenance                 36
Name: count, dtype: int64

In [39]:
df['ai_guess'].value_counts(normalize=True)

ai_guess
amenities                  0.332609
signage                    0.219565
access_restriction         0.210870
commercial_encroachment    0.158696
maintenance                0.078261
Name: proportion, dtype: float64